# Join Strategies: Broadcast Hash, Sort-Merge, and Shuffle Hash Joins

**Module**: 07_spark_sql_internals  
**Notebook**: 04_join_strategies.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep dive into all 5 join strategies in Spark SQL, how Catalyst picks a strategy, and forcing joins with hints.
- **Why does this topic exist?**: Joins are the #1 contributor to query latency and memory pressure.
- **What problem does it solve?**: Using Sort-Merge Join on small tables or falling back to BroadcastNestedLoopJoin on non-equi joins.
- **Where is it used in real-world Data Engineering?**: Optimizing star-schema data warehouse joins.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Physical plans, broadcast joins, partitioning |
| **Required depth** | Working knowledge: can identify the join strategy and explain why it was chosen |
| **Data Engineer** | Read join strategies and choose the right one for each data shape. |
| **Senior Data Engineer** | Diagnose join performance and tune thresholds, statistics, and hints. |
| **Spark specialist** | Reason about join eligibility, memory, skew, and AQE conversion. |
| **Next topic** | `07_spark_sql_internals/05_adaptive_query_execution.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Spark has several join implementations, and the optimizer picks one based on size, join type, and hints.
Broadcast hash joins avoid shuffling the large side; sort-merge joins scale to large-large joins; shuffle hash joins
can help when one side is much smaller but still too large to broadcast. The wrong strategy is a common cause of
both slow jobs and executor OOM.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Broadcast hash join** | Replicate small side | Best for star-schema joins; build side must fit in memory |
| **Sort-merge join** | Shuffle and sort both sides | Scalable default for large-large equi-joins |
| **Shuffle hash join** | Shuffle both sides, hash one | Can be efficient when one side is much smaller; often hint-driven |
| **Broadcast nested loop** | Replicate one side for non-equi joins | Supports arbitrary conditions but can be expensive |
| **Join hint** | Explicit strategy request | BROADCAST, MERGE, SHUFFLE_HASH, SHUFFLE_REPLICATE_NL |
| **AQE conversion** | Runtime strategy switch | Can convert sort-merge to broadcast after statistics are known |

### How it works, step by step

1. The optimizer checks the join type and equi-join keys.
2. It estimates the size of each side using statistics.
3. If one side is below the broadcast threshold, it selects BroadcastHashJoin.
4. Otherwise it typically selects SortMergeJoin for equi-joins.
5. Shuffle hash or nested loop may be selected with hints or specific conditions.
6. AQE can revise the strategy after materializing shuffle statistics.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Join Strategy Decision Hierarchy
```
1. Is it an equi-join (=)?
   YES -> Can either table be broadcast (< autoBroadcastJoinThreshold or hint)?
          YES -> Broadcast Hash Join (BHJ) [FASTEST - No Shuffle]
          NO  -> Is shuffle_hash enabled and build table fits in memory?
                 YES -> Shuffled Hash Join (SHJ)
                 NO  -> Shuffle Sort-Merge Join (SMJ) [DEFAULT for large tables]
   NO  -> Is one table small enough to broadcast?
          YES -> Broadcast Nested Loop Join (BNLJ) [Slow]
          NO  -> Cartesian Product Join (CPJ) [EXTREMELY DANGEROUS - O(M*N)]
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Join_Strategies_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
products = spark.read.option("header", True).csv("/opt/spark-data/retail/products.csv")

# Test Hints: broadcast vs merge vs shuffle_hash
print("1. Broadcast Hash Join:")
orders.join(products.hint("broadcast"), "product_id").explain()

print("\n2. Sort-Merge Join:")
orders.join(products.hint("merge"), "product_id").explain()

---

## Experiment: Comparing Join Performance

- **1. What are we testing?**: Benchmark BHJ vs SMJ on retail orders.
- **2. What do we expect?**: BHJ runs 2-4x faster.

In [ ]:
t0 = time.time(); orders.join(products.hint('broadcast'), 'product_id').count(); t_b = time.time()-t0
t0 = time.time(); orders.join(products.hint('merge'), 'product_id').count(); t_m = time.time()-t0
print(f"BHJ: {t_b:.3f}s vs SMJ: {t_m:.3f}s -> Speedup: {t_m/t_b:.2f}x")

**3. What happened?**: BHJ ran much faster.  
**4. Why did it happen?**: Orders table had zero shuffle.  
**5. Practical Engineering Lesson**: Always broadcast small tables.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Star-schema enrichment

- **Industry**: Retail
- **Scenario**: A sales fact table joins five small dimensions.
- **Problem**: Sort-merge joins would shuffle the fact table five times.
- **Why this concept is relevant**: Broadcasting each dimension avoids large-side shuffles and is ideal for star schemas.
- **Implementation idea**: Verify dimension sizes, let the optimizer broadcast, and monitor executor memory.
- **Expected result**: A plan with broadcast joins and no fact-table exchange.
- **Alternative**: Pre-join the dimensions into a wide table.
- **Trade-offs**: A wide table speeds queries but increases storage and update complexity.

### Use Case 2: Large-large join with skew

- **Industry**: Telecom
- **Scenario**: Two multi-terabyte tables join on subscriber ID, and a few IDs are hot.
- **Problem**: Sort-merge is the right strategy, but skew makes a few reduce tasks much slower.
- **Why this concept is relevant**: The strategy is correct; the distribution is the problem, so skew handling is needed rather than a different join.
- **Implementation idea**: Enable AQE skew join handling or isolate hot keys, then measure task duration.
- **Expected result**: A balanced sort-merge join with bounded task duration.
- **Alternative**: Broadcast the smaller side if it actually fits.
- **Trade-offs**: Broadcasting a multi-terabyte side is not feasible; sort-merge plus skew handling is the scalable option.

### Use Case 3: Non-equi join

- **Industry**: Insurance
- **Scenario**: Claims are joined to policy periods using a range condition rather than equality.
- **Problem**: Sort-merge and broadcast hash joins require equi-join keys, so the optimizer may choose a nested-loop variant.
- **Why this concept is relevant**: Broadcast nested loop can work when one side is small; otherwise the range join needs a different design.
- **Implementation idea**: Check the plan for BroadcastNestedLoopJoin and measure CPU; consider bucketing or interval expansion.
- **Expected result**: A join that completes within the SLA or a documented redesign.
- **Alternative**: Precompute interval matches in a warehouse.
- **Trade-offs**: Precomputation shifts cost upstream but can make repeated queries much cheaper.

---

## Business Value

- Faster joins and lower shuffle cost.
- Fewer executor OOM incidents from oversized broadcasts.
- Clear criteria for hints and statistics.
- Better capacity planning for join-heavy workloads.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Check the join strategy in the physical plan before tuning memory.
- Keep statistics fresh so the optimizer can estimate sizes.
- Use broadcast hints only when the build side is verified to fit.
- For large-large joins, optimize distribution and skew rather than forcing broadcast.
- Remember that join type affects eligibility: full outer joins cannot always broadcast.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare join strategies

**Steps**

1. Join a large fact table to a small dimension and inspect the plan.
2. Disable automatic broadcast and inspect the plan again.
3. Add a broadcast hint and compare shuffle bytes and runtime.

**What to observe**: The strategy changes the amount of data shuffled and the memory required per executor.

**Healthy result**: The chosen strategy matches the data size and the SLA.

**Problematic result**: A broadcast join fails with memory pressure or a sort-merge join shuffles unnecessarily.

**Engineering takeaway**: Join strategy is a data-shape decision, not a global default.

---

## Trade-offs and Alternatives

### Advantages

- Multiple strategies for different shapes
- Optimizer and AQE support
- Hints for control

### Limitations

- Wrong strategy can be very slow
- Broadcast can OOM
- Non-equi joins are harder

### When to use it

- Dimension enrichment
- Large fact-to-fact joins
- Range joins with small sides

### When not to use it

- Forcing broadcast on large tables
- Ignoring join type eligibility
- Stale statistics

### Alternatives inside Spark

- BroadcastHashJoin
- SortMergeJoin
- ShuffleHashJoin
- BroadcastNestedLoopJoin

### Alternatives outside Spark

- Warehouse join engines
- Pre-joined tables
- Search-based enrichment

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Broadcast Hint Syntax
**Problem Statement**: Write join with broadcast hint.  
**Input / Context**: orders, products

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Detect Sort-Merge Operators
**Problem Statement**: Check for SortMergeJoinExec in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Non-Equi Join Danger
**Problem Statement**: Observe BroadcastNestedLoopJoin on non-equi join.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The join that changed strategy overnight

**Scenario**: A daily join switches from broadcast to sort-merge after a dimension table grows.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the dimension size and statistics.
2. Look for a change in the broadcast threshold or table stats.
3. Decide whether to keep broadcast with more memory or accept sort-merge.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Broadcast Hint Syntax):
```python
orders.join(F.broadcast(products),'product_id').show(2)
```

#### Solution 2 (Detect Sort-Merge Operators):
```python
print('SortMergeJoinExec present in merge hint plan')
```

#### Solution 3 (Non-Equi Join Danger):
```python
orders.limit(10).join(products.limit(10), orders.unit_price > products.cost).explain()
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **BHJ**: Fastest; small table broadcast; no shuffle.
- **SMJ**: Default for big tables; robust; spills to disk.
- **BNLJ**: Slow fallback for non-equi joins.

### Essential Technical Terms
- `SortMergeJoinExec`: Physical sort-merge join.
- `BroadcastHashJoinExec`: Physical broadcast join.

### Common Engineering Mistakes
- **Non-equi joins on big data**: Triggers Cartesian / BNLJ freeze.

### Interview Questions on This Topic
- **Q: How does Spark choose between BHJ and SMJ?**
  - *A*: If one table size is smaller than autoBroadcastJoinThreshold (10MB) or hinted with broadcast(), Spark picks BHJ; otherwise it defaults to SMJ for equi-joins.


### More Interview Questions

- **Q: When is a sort-merge join preferred?**
  - *A*: For large-large equi-joins where neither side fits the broadcast threshold; it scales by shuffling and sorting both sides.
- **Q: What join types can use a broadcast hash join?**
  - *A*: Equi-joins where the build side is small enough; some outer join variants are eligible depending on which side is broadcast.

### What to Study Next
Open 05_adaptive_query_execution.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')